In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%load_ext tensorboard

In [ ]:
import gc
import sys
import logging
from pathlib import Path

In [ ]:
import mlflow

In [ ]:
import torch
import numpy as np

from ap_model_training import run


In [ ]:
_logger = logging.getLogger("ap_model_traning")
_logger.propagate = False
_logger.setLevel(logging.DEBUG)
for _handler in _logger.handlers:
    _logger.removeHandler(_handler)

_logging_formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S'"
)
_stream_handler = logging.StreamHandler()
_stream_handler.setLevel(logging.WARNING)
_stream_handler.setFormatter(_logging_formatter)

_logger.addHandler(_stream_handler)


In [ ]:
from ap_model_training.utils import MONAI_LOG_DIR

mlflow.pytorch.autolog()
port = 54598
mlflow_uri = f"file://{MONAI_LOG_DIR}"
_logger.info("Setting up mlflow with URI '%s'", mlflow_uri)
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("MONAI adaptive milling")

_file_handler = logging.FileHandler(MONAI_LOG_DIR / "training.log")
_file_handler.setLevel(logging.INFO)
_file_handler.setFormatter(_logging_formatter)
_logger.addHandler(_file_handler)

print(
    f"Run the following command to start:\n$mlflow ui --backend-store-uri {mlflow_uri} --port {port}\nThen navigate to:\nhttp://127.0.0.1:{port}"
)


In [ ]:
%%script false --no-raise-error
from utils import TENSORBOARD_LOG_DIR

logging.info("Setting up Tensorboard with log dir %s", TENSORBOARD_LOG_DIR)
%tensorboard --logdir $TENSORBOARD_LOG_DIR

In [ ]:
test_csv = "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/test.csv"
all_files_csv = (
    "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/all_files4.csv"
)


In [ ]:
gc.collect()
torch.cuda.empty_cache()
try:
    dist_matrix = np.asarray(
        [
            [0.0, 0.1, 0.1, 0.1, 0.1, 0.1],  # padding
            [0.1, 0.0, 0.5, 0.8, 0.6, 0.2],  # background
            [0.1, 0.5, 0.0, 0.8, 0.9, 1.0],  # lamella
            [0.1, 0.8, 0.8, 0.0, 0.7, 0.7],  # GIS
            [0.1, 0.6, 0.9, 0.7, 0.0, 0.9],  # crack
            [0.1, 0.2, 1.0, 0.7, 0.9, 0.0],  # void
        ],
        dtype=np.float32,
    )
    max_epochs = 120
    frozen_fraction = 0.2
    models_dir = Path.home() / "ap_model_training" / "models"
    run.run_training(
        # model_save_path = (
        #     Path.cwd().parent
        #     / "models"
        #     / "250317_234235_all_files_smp_efficientnet_b4_unetplusplus.pth"
        # )
        # mlflow_run_id = "f6453abe173949bf8564c868fa1a0528"
        # run.submit_validation_for_mlflow_run(
        #     mlflow_run_id,
        #     model_save_path,
        #     116,
        all_files_csv,
        models_dir=models_dir,
        model_name="fpn",  # "segresnet",
        loss_name="diceceloss",
        learning_rate=1e-5,
        epochs=max_epochs,
        image_size=1536,
        frozen_epochs=int(max_epochs * frozen_fraction),
        # model_kwargs={"encoder_weights": "advprop", "pretrained": True},
        loss_kwargs={
            "weights": (0.0, 1.0, 4.0, 3.0, 6.0, 2.0),
            "dist_matrix": dist_matrix,
        },
        gpu_number=0,
    )
finally:  # noqa: E722
    try:
        with torch.no_grad():
            torch.cuda.empty_cache()
    except Exception:
        _logger.error("Failed to clear CUDA cache", exc_info=True)
    try:
        mlflow.end_run()
    except:  # noqa: E722
        _logger.error("Failed to end MLFlow run", exc_info=True)